# Darts - Time-series Dense Encoder (TiDE)

In [22]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from darts import TimeSeries
import darts.metrics as metrics
from darts.models import TiDEModel
from darts.dataprocessing.transformers import Scaler, MissingValuesFiller

import torch
from pytorch_lightning.callbacks import EarlyStopping

Récupération du fichier + transformataion en timeseries

In [23]:
file = "datas/magnific7_1day.pkl"
series_dict = pd.read_pickle(file)

In [24]:
global_min_date = pd.Timestamp.max 
global_max_date = pd.Timestamp.min

for name, df in series_dict.items():
    df.index = pd.to_datetime(df.index)
    if not df.empty:
        global_min_date = min(global_min_date, df.index.min())
        global_max_date = max(global_max_date, df.index.max())

if global_min_date == pd.Timestamp.max or global_max_date == pd.Timestamp.min:
    raise ValueError("Aucune donnée valide trouvée dans series_dict pour déterminer l'index temporel global.")

full_global_time_index = pd.date_range(start=global_min_date, end=global_max_date, freq='B')

In [25]:
filler = MissingValuesFiller()

all_series = []
all_past_covariates = []

for name, df_original in series_dict.items():
    df_original.index = pd.to_datetime(df_original.index)

    df_reindexed = df_original.reindex(full_global_time_index)

    df_filled = df_reindexed.fillna(method='ffill').fillna(method='bfill')
    df_filled = df_filled.fillna(df_filled.mean())

    ts = TimeSeries.from_dataframe(df_filled, value_cols='close', fill_missing_dates=True, freq='B')
    ts = filler.transform(ts)
    all_series.append(ts)

    ts_past_covariates = TimeSeries.from_dataframe(df_filled, value_cols=['open', 'high', 'low', 'volume'], fill_missing_dates=True, freq='B')
    ts_past_covariates = filler.transform(ts_past_covariates)
    all_past_covariates.append(ts_past_covariates)

/tmp/ipykernel_11198/1016459720.py:11: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_filled = df_reindexed.fillna(method='ffill').fillna(method='bfill')
/tmp/ipykernel_11198/1016459720.py:11: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_filled = df_reindexed.fillna(method='ffill').fillna(method='bfill')
/tmp/ipykernel_11198/1016459720.py:11: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_filled = df_reindexed.fillna(method='ffill').fillna(method='bfill')
/tmp/ipykernel_11198/1016459720.py:11: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_filled = df_reindexed.fillna(method='ffill').fillna(method='bfill')
/tmp/ipy

### Séparation du jeu de données

In [26]:
train_series = []
val_series = []
train_past_covariates = []
val_past_covariates = []

for ts_target, ts_covariates in zip(all_series,  all_past_covariates):
    train_close, val_close = ts.split_before(0.8)
    train_series.append(train_close)
    val_series.append(val_close)

    train_cov, val_cov = ts_covariates.split_before(0.8)
    train_past_covariates.append(train_cov)
    val_past_covariates.append(val_cov)

In [27]:
close_scalers = []
past_cov_scalers = []

train_series_scaled = []
val_series_scaled = []
train_past_covariates_scaled = []
val_past_covariates_scaled = []

for i in range(len(train_series)):
    close_scaler = Scaler(StandardScaler())
    train_scaled = close_scaler.fit_transform(train_series[i])
    val_scaled = close_scaler.transform(val_series[i])

    close_scalers.append(close_scaler)
    train_series_scaled.append(train_scaled)
    val_series_scaled.append(val_scaled)

    past_cov_scaler = Scaler(StandardScaler())
    train_cov_scaled = past_cov_scaler.fit_transform(train_past_covariates[i])
    val_cov_scaled = past_cov_scaler.transform(val_past_covariates[i])

    past_cov_scalers.append(past_cov_scaler)
    train_past_covariates_scaled.append(train_cov_scaled)
    val_past_covariates_scaled.append(val_cov_scaled)

### Utilisation CUDA

In [28]:
if torch.cuda.is_available():
    print(f"CUDA est disponible ! Nombre de GPUs : {torch.cuda.device_count()}")
    gpu_devices = [0]
else:
    print("CUDA n'est pas disponible. Utilisation du CPU.")
    gpu_devices = 0

CUDA est disponible ! Nombre de GPUs : 1


In [31]:
early_stopper = EarlyStopping(
    monitor="train_loss", 
    patience=20,
    min_delta=0.05,
    mode='min'
)

# Le reste de pl_trainer_kwargs reste identique
pl_trainer_kwargs={
    "callbacks": [
        early_stopper
    ],
    "accelerator": "gpu",
    "devices": gpu_devices
}

## Modèle TiDE

In [32]:
model_tide = TiDEModel(
    model_name="tide",
    input_chunk_length=6,
    output_chunk_length=6,
    n_epochs=100,
    pl_trainer_kwargs=pl_trainer_kwargs,
    random_state=42,
    batch_size=32,
    hidden_size=128,
    num_encoder_layers=2,
    num_decoder_layers=2,
)

model_tide.fit(
    series=train_series_scaled,
    past_covariates=train_past_covariates_scaled,
    verbose=True,
)


number of `past_covariates` features is <= `temporal_width_past`, leading to feature expansion.number of covariates: 4, `temporal_width_past=4`.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name                | Type             | Params | Mode 
-----------------------------------------------------------------
0 | criterion           | MSELoss          | 0      | train
1 | train_criterion     | MSELoss          | 0      | train
2 | val_criterion       | MSELoss          | 0      | train
3 | train_metrics       | MetricCollection | 0      | train
4 | val_metrics         | MetricCollection | 0      | train
5 | past_cov_projection | _ResidualBlock   | 1.2 K  | train
6 | encoders            | Sequential       | 74.0 K | train
7 | decoders            | Sequential       | 90.8 K | train
8 | temporal_decoder    | _ResidualBlock   | 594    | train
9 | lookback_skip       | Line

Epoch 20: 100%|██████████| 696/696 [00:06<00:00, 107.85it/s, train_loss=0.0273]      


TiDEModel(output_chunk_shift=0, num_encoder_layers=2, num_decoder_layers=2, decoder_output_dim=16, hidden_size=128, temporal_width_past=4, temporal_width_future=4, temporal_hidden_size_past=None, temporal_hidden_size_future=None, temporal_decoder_hidden=32, use_layer_norm=False, dropout=0.1, use_static_covariates=True, model_name=tide, input_chunk_length=6, output_chunk_length=6, n_epochs=100, pl_trainer_kwargs={'callbacks': [<pytorch_lightning.callbacks.early_stopping.EarlyStopping object at 0x7f4277ead1c0>], 'accelerator': 'gpu', 'devices': [0]}, random_state=42, batch_size=32)

In [ ]:
def evaluate_model_historical_forecasts(model, 
                   train_series_scaled, val_series_scaled, target_scalers,
                   train_past_covariates_scaled=None, val_past_covariates_scaled=None, 
                   forecast_horizon=None):

    """
    Évalue un modèle Darts en utilisant la méthode historical_forecasts()
    sur les séries de validation, en gérant les covariables passées.

    Args:
        model: Le modèle Darts entraîné.
        train_series_scaled: Liste des séries d'entraînement mises à l'échelle (cible).
        val_series_scaled: Liste des séries de validation mises à l'échelle (cible).
        target_scaler: Le scaler utilisé pour les séries cibles (pour la dénormalisation).
        train_past_covariates_scaled: Liste des covariables passées d'entraînement mises à l'échelle (optionnel).
        val_past_covariates_scaled: Liste des covariables passées de validation mises à l'échelle (optionnel).
        past_cov_scaler: Le scaler utilisé pour les covariables passées (pour la dénormalisation, si besoin).
        forecast_horizon: L'horizon de prédiction désiré (n dans predict).
                          Si None, utilise output_chunk_length du modèle.

    Returns:
        pd.DataFrame: Un DataFrame contenant les métriques MAE, RMSE et MAPE
                      pour chaque série évaluée.
        list: Liste des prédictions dénormalisées pour chaque série.
    """

    if forecast_horizon is None:
        forecast_horizon = model.output_chunk_length

    performance = {}
    all_denormalized_predictions = []

    # Vérification de la présence de covariables passées
    use_past_covariates = train_past_covariates_scaled is not None and val_past_covariates_scaled is not None

    # Boucle sur chaque série
    for i in range(len(val_series_scaled)):
        current_train_series = train_series_scaled[i]
        current_val_series = val_series_scaled[i]

        #  Concatenation des séries d'entraînement et de validation
        full_series_for_backtest = current_train_series.append(current_val_series)

        current_past_covariates_for_backtest = None
        if use_past_covariates:
            current_train_past_cov = train_past_covariates_scaled[i]
            current_val_past_cov = val_past_covariates_scaled[i]
            current_past_covariates_for_backtest = current_train_past_cov.append(current_val_past_cov)

        #Définir le pout de départ pour historical_forecasts
        start_val_time = current_val_series.start_time()

        # Prédictions historiques
        try:
            if use_past_covariates:
                predictions_scaled = model.historical_forecasts(
                    series=full_series_for_backtest,
                    past_covariates=current_past_covariates_for_backtest,
                    start=start_val_time,
                    forecast_horizon=forecast_horizon,
                    stride=1,
                    retrain=False,
                    last_points_only=True,
                    verbose=False
                )
            else:
                predictions_scaled = model.historical_forecasts(
                    series=full_series_for_backtest,
                    start=start_val_time,
                    forecast_horizon=forecast_horizon,
                    stride=1,
                    retrain=False,
                    last_points_only=True,
                    verbose=False
                )        
        except Exception as e:
            print(f"Erreur lors de l'évaluation de la série {i}: {e}")
            predictions_scaled = TimeSeries.from_times_and_values(current_val_series.time_index[:0], np.array([]))

        # Dénormalisation des prédictions
        pred = target_scalers[i].inverse_transform(predictions_scaled)
        actual = target_scalers[i].inverse_transform(current_val_series)

        #  Alignement des séries sur la période actual 
        pred_aligned = pred.slice_intersect(actual)
        actual_aligned = actual.slice_intersect(pred)

        all_denormalized_predictions.append(pred_aligned)

        #  Calcul des métriques
        if len(actual_aligned) > 0 and len(pred_aligned) > 0:
            mae_value = metrics.mae(actual_aligned, pred_aligned)
            rmse_value = metrics.rmse(actual_aligned, pred_aligned)
            mape_value = metrics.mape(actual_aligned, pred_aligned)
        else:
            mae_value, rmse_value, mape_value = np.nan, np.nan, np.nan

        performance[f'Série {i}'] = {
            "MAE": mae_value,
            "RMSE": rmse_value,
            "MAPE": mape_value
        }
    
    return pd.DataFrame(performance).T, all_denormalized_predictions

In [ ]:
perf_df_historical_forecast, all_predictions_fot_plot = evaluate_model_historical_forecasts(
    model_tide,
    train_series_scaled,
    val_series_scaled,
    scalers,
    train_past_covariates_scaled,
    val_past_covariates_scaled,
    forecast_horizon=model_tft.output_chunk_length
)
print(perf_df_historical_forecast) 

### Visualisation :

In [ ]:
company_names = ['APPLE', 'MICROSOFT', 'AMAZON', 'ALPHABET', 'NVIDIA', 'META', 'TESLA']

# Pour chaque série (entreprise)
for idx in range(len(train_series)):
    # Récupérer les données d'entraînement et de validation
    train_data = train_series[idx]
    val_data = val_series[idx]
    
    pred_data = all_predictions_fot_plot[idx]

    mae_val = perf_df_historical_forecast.loc[f'Série {idx}', 'MAE']
    mape_val = perf_df_historical_forecast.loc[f'Série {idx}', 'MAPE']

    # Créer une nouvelle figure pour chaque entreprise
    plt.figure(figsize=(15, 6))

    # Tracer les séries temporelles
    train_data.plot(label='Entraînement', color='blue')
    val_data.plot(label='Validation (réel)', color='green')
    pred_data.plot(label='Prédiction (pas à pas)', color='red', linestyle='--')

    # Ajouter titre et légende avec métriques
    plt.title(f'Prédictions pour {company_names[idx]} (MAE: {mae_val:.2f}, MAPE: {mape_val:.2f}%)')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()